# 01 · KITTI-360 supplement (drive 0000)

Prepares what the rest of OCGF needs from the KITTI-360 starter dataset plus the official Velodyne scans:

- **scans** with each point's laser ring (computed on the raw scan order), cropped to the BEV region, stored in compact shards;
- **velodyne timestamps** and a **frame index**: which frames are usable, and which are sampled every 5 m of travel;
- a **quality report**: missing frames, image/scan alignment, ring sanity, LiDAR points drawn on images;
- a **dev sample** of about 50 consecutive frames for local CPU work.

## Before you run

1. **Settings → Accelerator:** None (CPU is enough). **Internet:** On.
2. **Add Input:** `pzelner6900/kitti-360-prepared-dataset`, your private `kitti360-3d-bboxes` and your private `kitti360-2d-semantics`.
3. The Velodyne download links come from `configs/data_sources/kitti360.yaml` in the repo.
4. For a quick trial set `MAX_FRAMES = 200` and run the cells. For the real run set `MAX_FRAMES = None` and use **Save Version → Save & Run All**.

In [ ]:
from pathlib import Path
import time

REPO_URL   = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF    = "p0/skeleton-geometry"   # branch, tag or commit; pin a commit for the final run
SEQUENCE   = "2013_05_28_drive_0000_sync"
INPUT_ROOT = Path("/kaggle/input")
OUT_DIR    = Path("/kaggle/working/kitti360_supplement")   # kept as the notebook output
SCRATCH    = Path("/kaggle/temp")                          # temporary, not kept
MODE       = "auto"   # how to read the 16 GB velodyne zip: auto | download | remote
MAX_FRAMES = None     # e.g. 200 for a quick trial; None for every scan
DEV_FRAMES = 50

CODE_DIR = SCRATCH / "OCGF"
TIMINGS = {}

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import shutil, subprocess, sys

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

SCRATCH.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))      # the editable install is only seen by new processes

from ocgf.utils.provenance import environment
print(environment())

## Inputs

Finds every component under `/kaggle/input` and stops with a clear message if one is missing.

In [ ]:
from ocgf.datasets import prepare_kitti360 as prep

cfg = prep.PrepConfig(sequence=SEQUENCE, input_root=INPUT_ROOT, out_dir=OUT_DIR, scratch_dir=SCRATCH,
                      sources=CODE_DIR / "configs" / "data_sources" / "kitti360.yaml",
                      mode=MODE, max_frames=MAX_FRAMES, dev_frames=DEV_FRAMES)
sources = prep.load_sources(cfg.sources, SEQUENCE)

t = time.time()
found = prep.stage_inputs(cfg)
TIMINGS["inputs"] = round(time.time() - t, 1)

## Velodyne zip and timestamps

`auto` downloads the zip to scratch if the disk has room (resuming if interrupted); otherwise it reads it in place over HTTP range requests. Either way the scans are read straight from the zip.

In [ ]:
t = time.time()
velodyne_zip, how = prep.open_zip(sources["velodyne"], cfg)
print("velodyne zip:", how)
prep.stage_velodyne_timestamps(cfg, velodyne_zip, sources)
TIMINGS["open_velodyne"] = round(time.time() - t, 1)

## Scans

For every scan: ring index on the raw point order, crop to x ∈ [−10, 80) m, y ∈ [−40, 40) m, encode (8 bytes per point), write to 500-frame shards.

In [ ]:
t = time.time()
scan_stats, profiles = prep.stage_scans(cfg, velodyne_zip)
TIMINGS["scans"] = round(time.time() - t, 1)

## Frame index

Usable = image, scan, pose, inside a semantic window, at least 50 m of future path, and image/scan aligned.

In [ ]:
t = time.time()
index = prep.stage_frame_index(cfg, found, scan_stats)
TIMINGS["frame_index"] = round(time.time() - t, 1)

## Report

In [ ]:
from IPython.display import Image, Markdown, display

t = time.time()
report = prep.stage_report(cfg, found, index, scan_stats, profiles)
TIMINGS["report"] = round(time.time() - t, 1)

display(Markdown("\n".join(l for l in report.read_text().splitlines() if not l.startswith("!["))))
for png in sorted(report.parent.glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))

## Dev sample

About 50 consecutive usable frames in one semantic window, with every component, laid out like KITTI-360.

In [ ]:
t = time.time()
prep.stage_dev_sample(cfg, found, index)
TIMINGS["dev_sample"] = round(time.time() - t, 1)

## Manifest

In [ ]:
prep.stage_manifest(cfg, found, index, sources, how, TIMINGS)
velodyne_zip.close()
sh(f'du -sh "{OUT_DIR}"/* | sort -h')

## After the run

- `/kaggle/working/kitti360_supplement` is this notebook's output. Later notebooks attach it with **Add Input → Your Work → this notebook**, or you can turn it into the private dataset `kitti360-d0000-supplement`.
- Download `dev_sample.zip` from the **Output** tab and unzip it to `C:\Users\nanin\data\kitti360_dev\` for local work.
- Send the summary of `report/report.md` back for review.